# 25 - Agentic RAG

Agentic RAG combines **RAG + agent decision-making**.

Normal RAG:
```text
question → retrieve → answer
```

Agentic RAG:
```text
question
 ↓
agent decides whether to search
 ↓
search
 ↓
inspect evidence
 ↓
enough evidence?
 ├─ yes → answer
 └─ no → search again / transform query
```

The RAG notebook you supplied provides retrieval, Chroma, citation-ready context, reranking, and evaluation; the agent notebook provides tools and the agent loop. This notebook combines those two foundations. fileciteturn19file6 fileciteturn19file0

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU langchain

## 1. Retrieval as a tool

The agent does not need the full vector database implementation exposed directly. Give it a clean search tool.

In [ ]:
from langchain.tools import tool

KNOWLEDGE = [
    'RAG retrieves relevant external information before generation.',
    'Chunk size affects retrieval quality.',
    'Reranking can improve the order of retrieved candidates.',
    'Hybrid retrieval combines semantic and keyword signals.',
]

@tool
def search_knowledge(query: str) -> str:
    'Search the demo knowledge base and return relevant evidence.'
    terms = set(query.lower().split())
    scored = []

    for text in KNOWLEDGE:
        score = sum(term in text.lower() for term in terms)
        scored.append((score, text))

    scored.sort(key=lambda x: x[0], reverse=True)
    return '\n'.join(text for score, text in scored[:3])

print(search_knowledge.invoke({'query': 'How can reranking improve retrieval?'}))

## 2. Agent decides when to retrieve

The important part is that retrieval is now an action the agent can choose.

It should not search for every trivial request, but it should search when evidence is needed.

In [ ]:
tools = [search_knowledge]
print([t.name for t in tools])

## 3. Add an evidence verifier

A verifier can judge whether retrieved evidence is sufficient before the agent answers.

In [ ]:
def evidence_sufficient(evidence, required_terms):
    evidence_lower = evidence.lower()
    return all(term.lower() in evidence_lower for term in required_terms)

evidence = search_knowledge.invoke({'query': 'reranking retrieval'})
print(evidence_sufficient(evidence, ['reranking']))

## 4. Multi-hop retrieval

Some questions need:
```text
search A
 ↓
use result from A to formulate B
 ↓
search B
 ↓
combine evidence
```

This is one of the strongest reasons to use an agent over fixed RAG.

## 5. Citations

Agentic RAG should still preserve source metadata.

The agent can decide **when** to retrieve, but the application should remain responsible for trustworthy source tracking.

## 6. Evaluation

Evaluate:
- whether search was used when needed
- unnecessary searches
- evidence sufficiency
- answer faithfulness
- task success
- latency/cost

## 7. Practice

1. Add two retrieval tools.
2. Create a multi-hop question.
3. Add a verifier.
4. Add a max-search limit.
5. Compare normal RAG vs agentic RAG.

## Evaluation

Improvement:
- query rewriting
- reranking
- evidence grading
- source diversity
- max-search budget
- explicit abstention
- citation validation.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/